# Estimating π with a Monte Carlo Simulation

## Problem and objective
The goal of this project is to estimate the mathematical constant π using random sampling. The method places points uniformly at random in a square and uses the proportion that land inside a quarter circle to estimate π. We will study how the number of sampled points affects the estimates and how much results vary between independent runs.

## Mathematical idea
Consider the unit square with coordinates from 0 to 1 on both axes. A quarter circle of radius 1 fits inside it. The square has area 1, while the quarter circle has area π/4. If points are sampled uniformly, the probability that a point falls inside the quarter circle is therefore π/4.

For a simulation with \(N\) points, let \(K\) be the number satisfying \(x^2+y^2 \leq 1\). The proportion \(K/N\) estimates the probability \(π/4\), giving the estimate

\[\hat{\pi}=4\frac{K}{N}.\]

## Questions for the experiment
- How does the estimation error change as the number of points increases?
- At the same point count, how much do estimates differ across independent runs?
- Do repeated-run summaries, such as the mean and standard deviation, support what we expect from the Law of Large Numbers?

We will investigate these questions using several point counts and repeated runs at each count. The results will be summarized with suitable error measures and visualizations. The simulation, metric calculations, and plotting will be written as reusable functions in notebook cells and applied consistently across the experiments.

## Assumptions and scope
The method assumes that sampled points are independent and uniformly distributed over the square. This is a numerical experiment: a larger sample tends to improve the estimate, but an individual run is not guaranteed to be closer to π. We will describe the observed results and limitations rather than claim that one run proves the method always behaves the same way.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The first simulation function

The function below runs one simulation. It generates \(N\) random points in the unit square. For each point \((x, y)\), it checks whether \(x^2+y^2 \leq 1\), which means the point is inside the quarter circle.

If \(K\) of the points are inside, the function returns \(4K/N\), our estimate of π. The optional `seed` lets us repeat a run with the same random points. Leaving the seed out gives a new random sample.

In [1]:
def estimate_pi(n_points, seed=None):
    if n_points <= 0:
        raise ValueError("n_points must be greater than zero.")

    rng = np.random.default_rng(seed)
    points = rng.random((n_points, 2))
    inside = points[:, 0] ** 2 + points[:, 1] ** 2 <= 1

    return 4 * inside.mean()

We can try the function once with 1,000 points. The seed makes this example repeatable; later, we’ll compare several point counts and repeated runs.

In [2]:
estimate_pi(n_points=1_000, seed=42)

np.float64(3.084)

## Repeated-run experiment

We will compare four sample sizes: 100, 1,000, 10,000, and 100,000 points. For each size, we will run the simulation 50 times. This gives us repeated estimates at each sample size so we can later compare their average accuracy and how much they vary.

Each run uses a different seed. This makes the experiment reproducible while giving each run a different random sample.

In [3]:
point_counts = [100, 1_000, 10_000, 100_000]
n_repeats = 50
results = {}

seed = 0
for n_points in point_counts:
    estimates = []
    for _ in range(n_repeats):
        estimates.append(estimate_pi(n_points, seed=seed))
        seed += 1
    results[n_points] = estimates

## Comparing the repeated estimates

For each point count, we have 50 estimates. We will summarize them with the mean estimate, the mean absolute error from π, and the sample standard deviation. The mean absolute error describes how far estimates are from π on average; the standard deviation describes how spread out they are around their own mean.

In [ ]:
def summarize_estimates(estimates):
    estimates = np.array(estimates)
    return {
        "mean_estimate": estimates.mean(),
        "mean_absolute_error": np.abs(estimates - np.pi).mean(),
        "standard_deviation": estimates.std(ddof=1)
    }


summary = {}
for n_points, estimates in results.items():
    summary[n_points] = summarize_estimates(estimates)

summary

## Expected variation

For each point, let \(I_i=1\) if it lands inside the quarter circle and \(I_i=0\) otherwise. Its probability of being inside is \(p=π/4\), and our estimate is \(\hat{\pi}=4(I_1+\cdots+I_N)/N\).

If the points are independent, the standard deviation of the estimate is:

\[4\sqrt{\frac{p(1-p)}{N}}\]

This gives the expected spread between runs for each point count. We can compare it with the standard deviation we measured from our 50 runs.

In [ ]:
p_inside = np.pi / 4
theoretical_spreads = {
    n: 4 * np.sqrt(p_inside * (1 - p_inside) / n)
    for n in point_counts
}
theoretical_spreads

## Visualizing the results

This plot compares average error and run-to-run variation at each point count. The dashed line shows the theoretical standard deviation. The horizontal axis uses a log scale because the point counts range from 100 to 100,000. Random results can fluctuate, so the lines may not decrease at every step.

In [ ]:
point_counts = sorted(summary)
errors = [summary[n]["mean_absolute_error"] for n in point_counts]
spreads = [summary[n]["standard_deviation"] for n in point_counts]
theoretical = [theoretical_spreads[n] for n in point_counts]

plt.plot(point_counts, errors, marker="o", label="Mean absolute error")
plt.plot(point_counts, spreads, marker="o", label="Standard deviation")
plt.plot(point_counts, theoretical, marker="s", linestyle="--", label="Theoretical standard deviation")
plt.xscale("log")
plt.xlabel("Points per run")
plt.ylabel("Value")
plt.title("Error and variation by sample size")
plt.legend()
plt.grid(True)
plt.show()

## Interpretation

In these runs, both the mean absolute error and the standard deviation decreased as the number of points increased. The mean absolute error went from 0.1216 with 100 points to 0.00413 with 100,000 points. The standard deviation went from 0.14876 to 0.00509, so the estimates were also more consistent at the larger sample sizes.

The average estimate moved closer to π as well: it was 3.1232 with 100 points and 3.14209 with 100,000 points. This pattern is consistent with the Law of Large Numbers. It does not mean that every individual run will be closer to π when more points are used. These results summarize 50 runs at each of four sample sizes, and other random runs could give somewhat different values.